# MSC Figure 5 — Anti-tautology simulation dataset

Upload the MSC model and YAML configuration; no Google Drive is required. This notebook produces a canonical encounter-level dataset for the Figure 5 R analysis.

## Scientific aim
The analysis tests whether simulated docking follows **model-used material mismatch** `d2_used`, rather than an undeclared direct pathway from authentic cargo identity to docking. `d2_true` is calculated from the authentic granule vector; `d2_used` is calculated from the vector supplied to the docking mechanism.

## Design
- Six variants, 100 parameter-generating seeds, 10,000 reservoir-sampled encounter opportunities per seed and variant.
- Expected output: 1,000,000 observations per variant; 6,000,000 total.
- Full, capture-only and projected-1D preserve the true-used mapping. Shuffled and random material disrupt it. Activity-only removes material terms from the capture rule.
- The notebook only produces raw and diagnostic data. The companion R script will fit seed-fixed-effect, seed-clustered logistic models, pooled interactions, Holm-adjusted TOST and odds ratios.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','pyyaml','pyarrow'],check=False)
import hashlib,json,os,shutil,time,zipfile,importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
print('Python',sys.version.split()[0],'numpy',np.__version__,'pandas',pd.__version__)

## Upload inputs
Upload `msc_model_feedback.py` (or `msc_model_feedback_used.py`) and the corresponding YAML file.

In [ ]:
from google.colab import files
WORK=Path('/content/msc_figure5');WORK.mkdir(exist_ok=True)
up=files.upload()
for n in up: shutil.move('/content/'+n,WORK/n)
py=sorted(WORK.glob('*.py')); yml=sorted(list(WORK.glob('*.yaml'))+list(WORK.glob('*.yml')))
assert py and yml,'Upload one model .py and one YAML config.'
ORIGINAL=py[0]; CONFIG=yml[0]
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
print(sha(ORIGINAL)[:16],ORIGINAL.name);print(sha(CONFIG)[:16],CONFIG.name)

## Canonical design
Run the smoke test first. It uses 3 seeds and 1,000 observations per seed/variant. Then set `SMOKE=False`, restart the runtime, and run every cell. The publication run uses 100 seeds and 10,000 observations per seed/variant.

In [ ]:
SMOKE=True
PLAN={'analysis':'MSC anti-tautology raw-data generation','N_SEEDS':100,'SEED_BASE':1000,'MAX_OBS_PER_RUN':10000,'variants':['full_msc_feedback','msc_capture_only','projected1d_feedback','shuffled_material_feedback','random_material_feedback','tag_only_feedback'],'equivalence_bound_logit':0.03,'correction':'Holm','model':'dock ~ d2_true or d2_used + T + M + C(seed)','inference_note':'R script performs all statistical inference'}
N_RUN=3 if SMOKE else PLAN['N_SEEDS']; MAX_OBS=1000 if SMOKE else PLAN['MAX_OBS_PER_RUN']
OUT=WORK/'figure5_output'
for x in ['01_inputs','02_raw','03_derived','04_r_input','05_logs']: (OUT/x).mkdir(parents=True,exist_ok=True)
print('SMOKE',SMOKE,'seeds',N_RUN,'max obs/run',MAX_OBS)

## Explicit, auditable model copy
A copy of the uploaded model is written to `01_inputs/msc_model_antitautology.py`. Only two documented changes are made: the simulation computes `d2_used` from `u_used` and `d2_true` from `u_true`; both values are stored in observation rows. The original uploaded model remains unchanged.

In [ ]:
src=ORIGINAL.read_text(encoding='utf-8')
old='d2 = material_distance2(u_used[g], theta[i], axes)'
new='d2_used = material_distance2(u_used[g], theta[i], axes)\n                d2_true = material_distance2(u_true[g], theta[i], axes)\n                d2 = d2_used'
assert src.count(old)==1,f'Expected one docking-distance line, found {src.count(old)}'
patched=src.replace(old,new,1)
oldrow='"d2": float(d2),'
newrow='"d2": float(d2),\n"d2_used": float(d2_used),\n"d2_true": float(d2_true),'
assert patched.count(oldrow)==2,f'Expected two event-row d2 fields, found {patched.count(oldrow)}'
patched=patched.replace(oldrow,newrow)
PATCHED=OUT/'01_inputs'/'msc_model_antitautology.py';PATCHED.write_text(patched,encoding='utf-8')
assert 'd2_true = material_distance2(u_true[g], theta[i], axes)' in patched
assert patched.count('d2_used') >= 3 and patched.count('d2_true') >= 3
shutil.copy2(ORIGINAL,OUT/'01_inputs'/ORIGINAL.name);shutil.copy2(CONFIG,OUT/'01_inputs'/CONFIG.name)
spec=importlib.util.spec_from_file_location('msc_antitautology',PATCHED);msc=importlib.util.module_from_spec(spec);spec.loader.exec_module(msc)
cfg=msc.load_config(str(CONFIG))
(OUT/'01_inputs'/'analysis_plan.json').write_text(json.dumps(PLAN,indent=2))
print('Patched model validated:',PATCHED.name,sha(PATCHED)[:16])

## Run simulations
Observations are recorded for every variant. The per-seed parameter draw is shared across variants, while the stochastic simulation seed is variant-specific. A checkpoint parquet file is updated every five seeds.


In [ ]:
def offset(v): return sum((j+1)*ord(c) for j,c in enumerate(v))%10000
frames=[];params_rows=[];t0=time.time();CK=OUT/'02_raw'/'encounters_checkpoint.parquet'
for s in range(N_RUN):
    seed=PLAN['SEED_BASE']+s;base=msc.sample_params(seed,cfg);base['max_obs_per_run']=MAX_OBS;params_rows.append(base)
    for v in PLAN['variants']:
        out=msc.simulate_one(base,seed=seed*100+offset(v),variant_name=v,record_obs=True)
        o=out['obs_df'].copy();assert len(o)==MAX_OBS,f'{v} seed {seed}: expected {MAX_OBS}, got {len(o)}';frames.append(o)
    if (s+1)%5==0:
        pd.concat(frames,ignore_index=True).to_parquet(CK,index=False);el=time.time()-t0;print(f'{s+1}/{N_RUN} | {el/60:.1f} min | ETA {el/(s+1)*(N_RUN-s-1)/60:.1f} min',flush=True)
raw=pd.concat(frames,ignore_index=True);raw.to_parquet(OUT/'02_raw'/'encounter_observations_all_variants.parquet',index=False)
pd.DataFrame(params_rows).to_csv(OUT/'02_raw'/'parameter_realizations.csv',index=False)
print('Rows:',len(raw))

## Validate the anti-tautology manipulation
The checks below are prerequisites for the Figure 5 claims. In preserving variants `d2_true` and `d2_used` must coincide. In shuffled/random variants they must diverge. `d2` is retained as an alias of the model-used mismatch for compatibility with earlier outputs.

In [ ]:
req=['seed','variant','T','M','d2','d2_used','d2_true','active_synapse','dock'];assert set(req).issubset(raw.columns)
diag=[]
for v,g in raw.groupby('variant'):
    diag.append({'variant':v,'n_encounters':len(g),'n_seeds':g.seed.nunique(),'n_docks':int(g.dock.sum()),'docking_rate':g.dock.mean(),'max_abs_d2_minus_used':float(np.abs(g.d2-g.d2_used).max()),'cor_true_used':float(g.d2_true.corr(g.d2_used)),'mean_abs_true_minus_used':float(np.abs(g.d2_true-g.d2_used).mean())})
diag=pd.DataFrame(diag).sort_values('variant');display(diag)
expected=N_RUN*MAX_OBS
assert (diag.n_encounters==expected).all() and (diag.n_seeds==N_RUN).all() and (diag.n_docks>0).all()
assert (diag.max_abs_d2_minus_used<1e-12).all()
pres=['full_msc_feedback','msc_capture_only','projected1d_feedback'];dis=['shuffled_material_feedback','random_material_feedback']
assert (diag.set_index('variant').loc[pres,'cor_true_used'].sub(1).abs()<1e-12).all()
assert (diag.set_index('variant').loc[dis,'cor_true_used'].abs()<0.95).all()
diag.to_csv(OUT/'03_derived'/'Figure5_true_used_separation_diagnostics.csv',index=False)
print('All anti-tautology design checks passed.')

## Stage R inputs, manifest and ZIP
The R script should read the parquet file and `analysis_plan.json` from `04_r_input/`. The raw parquet file is intentionally retained rather than expanded into a large CSV; it preserves types, is smaller, and is directly supported by R package `arrow`.

In [ ]:
RIN=OUT/'04_r_input'
for folder,name in [(OUT/'02_raw','encounter_observations_all_variants.parquet'),(OUT/'02_raw','parameter_realizations.csv'),(OUT/'03_derived','Figure5_true_used_separation_diagnostics.csv'),(OUT/'01_inputs','analysis_plan.json')]:shutil.copy2(folder/name,RIN/name)
checks={'smoke_run':SMOKE,'n_seeds':N_RUN,'max_obs_per_run':MAX_OBS,'expected_per_variant':expected,'total_rows':len(raw),'expected_total_rows':expected*len(PLAN['variants']),'variants':PLAN['variants']}
pd.DataFrame([checks]).to_csv(OUT/'05_logs'/'Figure5_run_checks.csv',index=False)
manifest={'figure':'Figure 5','generated_utc':pd.Timestamp.utcnow().isoformat(),'plan':PLAN,'checks':checks,'original_model_sha256':sha(ORIGINAL),'patched_model_sha256':sha(PATCHED),'config_sha256':sha(CONFIG),'output_sha256':{str(p.relative_to(OUT)):sha(p) for p in OUT.rglob('*') if p.is_file()}}
(OUT/'MANIFEST_figure5.json').write_text(json.dumps(manifest,indent=2))
(OUT/'README.md').write_text('# MSC Figure 5 reproducible bundle\n\nThis bundle contains an explicitly patched copy of the uploaded model, raw all-variant encounter observations, parameter realizations, anti-tautology manipulation diagnostics, analysis plan, hashes, and R-ready inputs. The patch adds d2_true and d2_used only; it does not change the docking rule.\n')
zp=Path('/content/MSC_Figure5_reproducible_bundle.zip')
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file():z.write(f,f.relative_to(OUT.parent))
print(zp.name,f'{zp.stat().st_size/1e6:.1f} MB');files.download(str(zp))